In [1]:
!git clone https://github.com/riyamaurya86/crowd-counting-partB.git

Cloning into 'crowd-counting-partB'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 149 (delta 6), reused 9 (delta 2), pack-reused 134 (from 1)
Receiving objects: 100% (149/149), 57.81 MiB | 47.02 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [2]:
%cd crowd-counting-partB

/kaggle/working/crowd-counting-partB


In [3]:
import os
import torch
from torch.utils.data import DataLoader

from src.datasets.shanghai_partb import ShanghaiPartBDataset
from src.models.csrnet_dual import CSRNet_Dual
from src.engine.trainer import train_one_epoch, validate, save_checkpoint
from src.losses.mse import get_mse_loss
from src.utils.seed import set_seed

In [4]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
dataset_path = "/kaggle/input/datasets/tthien/shanghaitech-with-people-density-map/ShanghaiTech/part_B"

train_dataset = ShanghaiPartBDataset(dataset_path, mode="train", crop_size=256)
test_dataset = ShanghaiPartBDataset(dataset_path, mode="test")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 400
Test size: 316


In [6]:
model = CSRNet_Dual(pretrained=True).to(device)

criterion = get_mse_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 200MB/s] 


In [8]:
num_epochs = 50
best_mae = float("inf")

os.makedirs("checkpoints/csrnet_dual", exist_ok=True)

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_results = validate(model, test_loader, criterion, device)

    print(f"Train Loss: {train_loss:.6f}")
    print(f"Val MAE: {val_results['mae']:.2f}")
    print(f"Val RMSE: {val_results['rmse']:.2f}")
    print(f"Val PSNR: {val_results['psnr']:.2f}")
    print(f"Val SSIM: {val_results['ssim']:.4f}")

    # Save best model
    if val_results["mae"] < best_mae:
        best_mae = val_results["mae"]
        best_results = val_results.copy()
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_mae,
            "checkpoints/csrnet_dual/best_model.pth"
       )


Epoch [1/50]


Validating: 100%|██████████| 316/316 [00:47<00:00,  6.65it/s]


Train Loss: 0.000001
Val MAE: 76.20
Val RMSE: 114.50
Val PSNR: 30.06
Val SSIM: 0.6574

Epoch [2/50]


Validating: 100%|██████████| 316/316 [00:47<00:00,  6.63it/s]


Train Loss: 0.000000
Val MAE: 51.62
Val RMSE: 83.38
Val PSNR: 30.19
Val SSIM: 0.6806

Epoch [3/50]


...

Train Loss: 0.000000
Val MAE: 31.12
Val RMSE: 38.13
Val PSNR: 31.87
Val SSIM: 0.7551

Epoch [49/50]


Validating: 100%|██████████| 316/316 [00:41<00:00,  7.64it/s]


Train Loss: 0.000001
Val MAE: 22.80
Val RMSE: 32.71
Val PSNR: 31.93
Val SSIM: 0.7639

Epoch [50/50]


Validating: 100%|██████████| 316/316 [00:41<00:00,  7.65it/s]

Train Loss: 0.000000
Val MAE: 21.44
Val RMSE: 38.12
Val PSNR: 32.07
Val SSIM: 0.7709


In [9]:
import pandas as pd

results_df = pd.DataFrame([best_results])
results_df.to_csv("results/csrnet_dual_metrics.csv", index=False)

print("Saved results.")

Saved results.


In [10]:
import shutil

shutil.copy(
    "checkpoints/csrnet_dual/best_model.pth",
    "/kaggle/working/csrnet_dual_best_model.pth"
)

print("Checkpoint copied to working directory.")

Checkpoint copied to working directory.


In [11]:
from src.utils.visualization import visualize_predictions

fixed_indices = [165, 173, 33, 78, 93]

visualize_predictions(
    model,
    test_dataset,
    device,
    save_dir="results/qualitative_results/csrnet_dual",
    indices=fixed_indices
)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].


Saved visualizations to results/qualitative_results/csrnet_dcn
